# Problem 1 (programming parts). A Tiny Causal Language Model, End to End

**Day 1 · Problem 1 · Parts 1.11–1.13 (25 of the problem's 90 points).**
The shared setup, the theory Parts 1.1–1.10, and Part 1.14 are in `theory/p01.md`; read its shared setup first.
The model, mask, positional table, shift, and perplexity used below are exactly the ones defined there.
Run the cells in order: Part 1.12 uses Part 1.11's functions, and Part 1.13 uses both.

**Allowed tools.** Python's standard library, NumPy, and PyTorch only, on a CPU. No web access, downloads, external datasets, model hubs, pretrained weights, or checkpoints you did not create in the same run.
**Banned APIs (zero points for the part that uses them):** `torch.nn.MultiheadAttention`, `torch.nn.functional.scaled_dot_product_attention`, and every `torch.nn.Transformer*` class.
Build attention from `nn.Linear`, tensor operations, and `torch.softmax`.

**Data.** `data/p01_corpus.json` holds the Problem 1 corpus (Part 1.14 states how it was generated): `"train"` is a list of 256 rows and `"val"` a list of 64 rows, each row 13 token IDs in `0..23`.
The cell below loads them as `train_tokens` `(256, 13)` and `val_tokens` `(64, 13)`, both `torch.int64`.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()

corpus = json.loads((DATA_DIR / "p01_corpus.json").read_text(encoding="utf-8"))
train_tokens = torch.tensor(corpus["train"], dtype=torch.int64)
val_tokens = torch.tensor(corpus["val"], dtype=torch.int64)
VOCAB_SIZE, D_MODEL, N_HEADS, N_LAYERS, D_FF, MAX_LEN = 24, 32, 4, 2, 64, 12
print(train_tokens.shape, val_tokens.shape)


## Part 1.11 (5 points)

**Type:** programming · **Answer form:** code · Coding allowed.

(a) Write `causal_mask(n)` returning a `torch.bool` tensor of shape `(n, n)` whose entry `[i, j]` is `True` exactly when `j <= i` (`True` means *allowed*).

(b) Write `masked_softmax(scores, allowed)`: `scores` has shape `(..., n, n)` and `allowed` is a Boolean tensor that broadcasts to it.
Replace every forbidden score by $-\infty$ **before** a softmax over the last axis and return the weights (same shape as `scores`).
Raise `ValueError` if any row has no allowed entry.

(c) Probe. Build the `float64` scores `S` of shape `(4, 4)` with `S[i, j] = ln(j + 1)` for every `i`, the value column `Vcol = [[1.], [2.], [3.], [4.]]` (`float64`), and compute `out = masked_softmax(S, causal_mask(4)) @ Vcol`, shape `(4, 1)`.
Then set `S[0, 3] = 100.0` (a forbidden entry), recompute, and check that `out` is unchanged exactly.

**Required prints and assertions.** Print `out[:, 0]`.
Assert the shape and dtype of `causal_mask(4)`, that every weight row sums to `1` (tolerance `1e-12`), that forbidden weights are exactly `0`, that the forbidden-score change leaves `out` bit-for-bit equal, and that an all-`False` row raises `ValueError`.
**Answer:** print `float(out[2, 0])` to six decimal places.

In [ ]:
def causal_mask(n):
    raise NotImplementedError


def masked_softmax(scores, allowed):
    raise NotImplementedError


# Part 1.11 probe, prints, and assertions.


## Part 1.12 (10 points)

**Type:** programming · **Answer form:** code · Coding allowed.

Write `class CausalSelfAttention(nn.Module)` implementing the shared setup's MHA with the causal mask:

* `__init__(self, d_model, n_heads)` raises `ValueError` unless `d_model % n_heads == 0`, then creates **in this order** `self.q_proj`, `self.k_proj`, `self.v_proj`, `self.o_proj`, each `nn.Linear(d_model, d_model)` (with bias), and creates no other parameters.
* `forward(x)` takes `x` of shape `(B, n, d_model)` with `n <= 12`, a floating dtype, and returns `(B, n, d_model)`.
  Split heads exactly as the shared setup states (`.view(B, n, n_heads, d_h).transpose(1, 2)` gives head `r` the feature columns `r*d_h ... (r+1)*d_h - 1`), scale scores by `1/sqrt(d_h)`, use `causal_mask` and `masked_softmax` from Part 1.11, merge heads with the inverse reshape, and apply `o_proj`.

**Probe.** Run `torch.manual_seed(SEED)` immediately before `attn = CausalSelfAttention(32, 4)`.
Set `x = torch.randn(2, 12, 32, generator=torch.Generator().manual_seed(SEED))` and `out = attn(x)`.
Then `x2 = x.clone(); x2[:, 7:] += 1.0` and `out2 = attn(x2)`.

**Required prints and assertions.**
`n_params = sum(p.numel() for p in attn.parameters())` equals your Part 1.4(a) count at `d = 32`;
`out.shape == (2, 12, 32)`;
`(out2[:, :7] - out[:, :7]).abs().max() == 0` exactly (positions before 7 cannot see positions 7–11);
`(out2[:, 7:] - out[:, 7:]).abs().max() > 0`;
`CausalSelfAttention(30, 4)` raises `ValueError`.
**Answer:** print `probe_value = float(out.detach().abs().mean())` to six decimal places.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


# Part 1.12 probe, prints, and assertions.


## Part 1.13 (10 points)

**Type:** programming · **Answer form:** code · Coding allowed.

(a) Write `sinusoidal_table(L, d)` returning the `float32` table of shape `(L, d)` defined in the shared setup, and `shift_for_causal_lm(tokens)` returning `(inputs, targets) = (tokens[:, :-1], tokens[:, 1:])`.

(b) Write `class TinyCausalLM(nn.Module)` implementing the shared setup's model $\mathcal M$ with constructor `TinyCausalLM()` (the sizes are fixed: `VOCAB_SIZE`, `D_MODEL`, `N_HEADS`, `N_LAYERS`, `D_FF`, `MAX_LEN` above).
Create submodules **in this order**: `self.tok = nn.Embedding(24, 32)`; a buffer `self.pe` registered with `register_buffer("pe", sinusoidal_table(12, 32))`; `self.blocks = nn.ModuleList(...)` of 2 blocks, where each block creates, in order, `ln1 = nn.LayerNorm(32)`, `attn = CausalSelfAttention(32, 4)`, `ln2 = nn.LayerNorm(32)`, `ff1 = nn.Linear(32, 64)`, `ff2 = nn.Linear(64, 32)` and computes the pre-norm block of the shared setup; then `self.ln_f = nn.LayerNorm(32)` and `self.head = nn.Linear(32, 24)`.
`forward(ids)` takes `int64` IDs `(B, n)` with `n <= 12`, adds `self.pe[:n]` to the token embeddings, runs the blocks, `ln_f`, and `head`, and returns logits `(B, n, 24)`.

(c) Train. Run `torch.manual_seed(SEED)` immediately before `model = TinyCausalLM()`; then `optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)`.
Using `shift_for_causal_lm(train_tokens)`, run exactly **150** full-batch updates in train mode; each update runs, in this order, `optimizer.zero_grad(set_to_none=True)`, forward, `loss = F.cross_entropy(logits.reshape(-1, 24), targets.reshape(-1))`, `loss.backward()`, `optimizer.step()`; record every loss in a list `losses`.
No scheduler, clipping, dropout, or early stopping.

(d) Evaluate. In eval mode under `torch.no_grad()`, compute the validation perplexity (shared setup; this corpus has no padding) on `shift_for_causal_lm(val_tokens)` **before** training (`val_ppl_before`, right after construction) and **after** training (`val_ppl_after`), and the 12 per-position validation perplexities after training (position `i` averages the cross-entropy over the 64 validation rows at output position `i` only).

**Required prints and assertions.**
`sum(p.numel() for p in model.parameters() if p.requires_grad)` equals your Part 1.5 answer, and `pe` is not among the parameters;
`model.pe[0, 0::2]` is all `0` and `model.pe[0, 1::2]` all `1`;
logits shape `(64, 12, 24)` on the validation inputs;
`len(losses) == 150` and `losses[-1] < 0.25 * losses[0]`;
`val_ppl_after < val_ppl_before`;
`1.05 < val_ppl_after < 1.20`;
the position-0 per-position perplexity is the largest of the 12.
In at most two sentences, explain the last certificate using Part 1.14(a).
**Answer:** print `val_ppl_after` to three decimal places.

In [ ]:
def sinusoidal_table(L, d):
    raise NotImplementedError


def shift_for_causal_lm(tokens):
    raise NotImplementedError


class TinyCausalLM(nn.Module):
    def __init__(self):
        super().__init__()
        raise NotImplementedError

    def forward(self, ids):
        raise NotImplementedError


# Part 1.13 construction, training, evaluation, prints, and assertions.


## Your explanation for Part 1.13 (at most two sentences)

*Replace this line.*

Continue with Part 1.14 in `theory/p01.md`.